# Setup

In [ ]:
!pip install demoji
#!pip install -U spacy
!pip install fasttext
!wget "https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.bin" # Download fasttext pretrained language id model

     |████████████████████████████████| 42 kB 1.1 MB/s 
     |████████████████████████████████| 68 kB 3.4 MB/s 
  Using cached pybind11-2.9.2-py2.py3-none-any.whl (213 kB)
  Created wheel for fasttext: filename=fasttext-0.9.2-cp37-cp37m-linux_x86_64.whl size=3139990 sha256=f3ecec06b15b6fdbffacdb7845ef716d09802205e3f4afb8b1b9b4808474e68e
  Stored in directory: /root/.cache/pip/wheels/4e/ca/bf/b020d2be95f7641801a6597a29c8f4f19e38f9c02a345bab9b
Successfully built fasttext
--2022-04-05 15:15:46--  https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.bin
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 172.67.9.4, 104.22.75.142, 104.22.74.142, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|172.67.9.4|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 131266198 (125M) [application/octet-stream]
Saving to: ‘lid.176.bin’

lid.176.bin         100%[===================>] 125.18M  58.1MB/s    in 2.2s    

2022-04-05 15:15:49 (58.1 

In [ ]:
# Imports
import glob
import os
import pandas as pd
import numpy as np
import regex as re
import html
import random
import time

import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

# import spacy
# from spacy.lang.en import English
# from spacy.attrs import ORTH

from nltk.tokenize import sent_tokenize
from nltk.tokenize import word_tokenize
from nltk.tag.stanford import StanfordPOSTagger
from nltk.data import load

import torch
import torch.nn as nn
from collections import Counter

import fasttext

import demoji

%load_ext autoreload
%autoreload 2

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# POS Setup

## Filipino POS Tagger

In [ ]:
%cd /content
!ls

/content
drive  lid.176.bin  sample_data


In [ ]:
!git clone https://github.com/jcblaisecruz02/filipino-pos.git

Cloning into 'filipino-pos'...
remote: Enumerating objects: 28, done.
remote: Total 28 (delta 0), reused 0 (delta 0), pack-reused 28
Unpacking objects: 100% (28/28), done.


In [ ]:
MODEL_DIR = "/content/filipino-pos/checkpoint"

In [ ]:
'''
Implements a simple LSTM Tagging model.
'''

class LSTMTagger(nn.Module):
    def __init__(self, word_vocab_sz, tag_vocab_sz, embedding_dim, hidden_dim, dropout=0.5, bidirectional=True, num_layers=1, recur_dropout=0.1):
        super(LSTMTagger, self).__init__()
        self.embedding = nn.Embedding(word_vocab_sz, embedding_dim)
        self.rnn = nn.LSTM(embedding_dim, hidden_dim, bidirectional=bidirectional, num_layers=num_layers, dropout=recur_dropout if num_layers > 1 else 0.0)
        self.fc1 = nn.Linear(hidden_dim * 2 if bidirectional else hidden_dim, tag_vocab_sz)
        self.dropout = nn.Dropout(dropout)

    def init_hidden(self, bs):
        params = next(self.parameters())
        hidden_dim = self.rnn.hidden_size
        directions = 2 if self.rnn.bidirectional else 1
        layers = self.rnn.num_layers

        hidden = params.new_zeros(layers * directions, bs, hidden_dim)
        cell = params.new_zeros(layers * directions, bs, hidden_dim)
        return hidden, cell

    def forward(self, x):
        msl, bs = x.shape
        out = self.embedding(x)
        hidden, cell = self.init_hidden(bs)
        out, (hidden, cell) = self.rnn(out, (hidden, cell))

        out = self.dropout(out)
        out = self.fc1(out)
        return out

# UTILS
# Normalizes quotes and lowercases a string
def normalize(line): return line.replace('“', '"').replace('”', '"').replace("‘", "'").replace("’", "'").lower()

# Indexes, truncates, then pads
def proc_set(s, word2idx, word_vocab, msl=128):
    proc = []
    for line in s:
        line = [word2idx[w if w in word_vocab else '<unk>'] for w in line.split()][:msl]
        if len(line) < msl: line += [word2idx['<pad>'] for _ in range(msl - len(line))]
        proc.append(line)
    return proc

def produce_vocab(words, min_freq=2):
    vocab = []
    for line in words: vocab.extend(line.split())
    counts = dict(Counter(vocab))
    vocab = [word for word in counts.keys() if counts[word] >= min_freq]
    vocab = set(vocab + ['<unk>', '<pad>'])
    idx2word = list(vocab)
    word2idx = {idx2word[i]:i for i in range(len(idx2word))}
    return vocab, idx2word, word2idx

def init_weights(m):
    for name, param in m.named_parameters():
        nn.init.normal_(param.data, mean=0, std=0.1)

# Categorical accuracy
def accuracy(out, y, tag2idx):
    msl, bs = y.shape
    acc = 0
    with torch.no_grad():
        preds = out.argmax(2)
        for i in range(bs):
            nz = (y[:,i] != tag2idx['<pad>']).sum().item()
            acc += (preds[:,i][:nz] == y[:,i][:nz]).float().mean().item()
    acc /= bs
    return acc

# Process one example string then returns a prediction
def predict(s, word2idx, idx2tag, word_vocab, msl, model):
    # Convert to input
    xs = normalize(s)
    l = len(xs.split())
    xs = proc_set([xs], word2idx, word_vocab, msl=msl)
    xs = torch.LongTensor(xs)
    # print(xs.size())
    
    # Produce prediction
    with torch.no_grad():
        out = model(xs)
    out = out.squeeze(1).argmax(1)
    preds = list(out[:l].numpy())

    preds = [idx2tag[ix] for ix in preds]
    return preds

# Process batch of string then returns a prediction
def predict_batch(sentences, word2idx, idx2tag, word_vocab, msl, model):

    bs = len(sentences)

    # Convert to input
    xs = [normalize(s) for s in sentences]
    lengths = [len(x.split()) for x in xs]
    xs = [proc_set([x], word2idx, word_vocab, msl=msl) for x in xs]
    xs = torch.LongTensor(np.array(xs)).squeeze().transpose(1, 0)
    word_tokens = [s.split() for s in sentences]

    # load data to gpu
    xs = xs.to(device)

    # Produce predictions
    with torch.no_grad():
        out = model(xs).argmax(2)
        
    # Process predictions
    pos = []
    preds = []
    for i in range(bs):
        sent = out[:, i][:lengths[i]]
        #print("SENTENCE: ", sent)
        sent_tags_pos = [] 
        sent_tags = ""
        #print(len(word_tokens[i]), len(sent))
        for index, ix in enumerate(sent.tolist()):
            tag = idx2tag[ix]
            # if '_' not in tag: # if not compound tags
            #     tag = tag
            
            # sent_tags += f"{tag} "
            # sent_tags.append((idx2word[index], tag))
            sent_tags_pos.append((word_tokens[i][index], tag))
            sent_tags += f"{tag} "
        
        #sent_tags = [sent_tags[j] if '_' in sent_tags[j] else sent_tags[j][:2] for j in range(len(sent_tags))]
        pos.append(sent_tags_pos) # remove trailing whitespaces
        preds.append(sent_tags.strip()) # remove trailing whitespaces
          
    return pos, preds

def batch(iterable, n=1):
    l = len(iterable)
    for ndx in range(0, l, n):
        yield iterable[ndx:min(ndx + n, l)]

def batch_tag_fil(sentences, bs):
    preds = []
    pos = []
    count = 0
    n_sentences = len(sentences)

    for x in batch(sentences, bs):
        batch_pos, batch_preds  = predict_batch(x, word2idx, idx2tag, word_vocab, msl, model)
        for pred in batch_preds:
            preds.append(pred)
        for pos_tag in batch_pos:
            pos.append(pos_tag) 
        count += len(x)
        print(f"Progress: {(count/n_sentences)*100}%")

    
    return preds, pos

In [ ]:
# Load the vocabularies
with open(MODEL_DIR + '/settings.bin', 'rb') as f:
    word_vocab, word2idx, idx2word, tags_vocab, tag2idx, idx2tag, msl, embedding_dim, hidden_dim, dropout, bidirectional, num_layers, recur_dropout = torch.load(f)

# Produce a blank model
model = LSTMTagger(word_vocab_sz=len(word_vocab), 
                    tag_vocab_sz=len(tags_vocab), 
                    embedding_dim=embedding_dim, 
                    hidden_dim=hidden_dim, 
                    dropout=dropout,
                    num_layers=num_layers,
                    recur_dropout=recur_dropout,
                    bidirectional=bidirectional)

# Load checkpoints and put the model in eval mode
with open(MODEL_DIR + '/model.bin', 'rb') as f:
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    print(f"DEVICE: {device}")
    model.load_state_dict(torch.load(f, map_location=device))
    model.to(device)
    model.eval();

DEVICE: cuda:0


## English POS Tagger

In [ ]:
def batch_tag_eng(sentences):
    '''
    Returns a list of strings of POS tags for every item in sentences.
    Iterates per sentence and CPU-based.
    '''
    pos = []
    preds = []
    for s in sentences:
        res = nltk.pos_tag(word_tokenize(s))
        pos.append(res)

        processed_tags = ""
        for word_tag_pair in res:
            word, tag = word_tag_pair
            processed_tags += f"{tag} "

        # clean non alphbet and trailing whitespaces
        processed_tags = re.sub('[^a-zA-Z]+', ' ', processed_tags).strip()
        preds.append(processed_tags)
    
    #print(preds, pos)
    return preds, pos

# Language Identfication Setup

In [ ]:
class LanguageIdentification:
    """
    Class object for Language Identification
    """

    def __init__(self):
        """
        Initializing class objects
        """
        pretrained_lang_model = "lid.176.bin"
        self.model = fasttext.load_model(pretrained_lang_model)

    def predict_lang(self, text):
        """
        Attributes
        text: type str, the sentence for which language is to be identified
        """
        predictions = self.model.predict(text, k=1) # top 1 matching languages
        return predictions
    
    def predict_lang_list(self, text_list):
        """
        Attributes
        text_list: type list, the list of sentences for which language is to be identified

        Returns 2D list of predicted language and prediction probability
        """
        langs = []
        probs = []
        for text in text_list:
            lang, prob = self.model.predict(text.replace("\n", ""), k=1) # top 1 matching languages
            lang_tag = lang[0][-2:]
            if lang_tag == 'en' and prob < 0.8:
                langs.append("tl")
                probs.append("-1") # -1 means overridden
            else:
                langs.append(lang_tag)
                probs.append(prob[0])

        return langs, probs

In [ ]:
detector = LanguageIdentification()
langs, probs = np.array(detector.predict_lang_list(['ang ganda naman ng drawing mo!', 'I don\'t want to talk to you.', '- MDM , GMA News']))
langs, probs

(array(['tl', 'en', 'tl'], dtype='<U32'),
 array(['0.9725924730300903', '0.9999812841415405', '-1'], dtype='<U32'))

# Setup tokenizer

In [ ]:
language = "english"
tokenizer = load(f"tokenizers/punkt/{language}.pickle")
# df = pd.read_json('raw cohfie/3.json')

In [ ]:
# df['text'].apply(tokenizer.tokenize).explode().to_frame().reset_index(drop=True).style

# tokenizer.tokenize(df['text']) # usage

# Processing

In [ ]:
def normalize_abbreviations(text):
    '''
    Remove dot from abbreviation and normalize to one form\.
    '''
    text = re.sub(r"(?<!\S0-9)(A|a)\.(M|m)\.", " AM ", text) # A\.M\.
    text = re.sub(r"(?<!\S0-9)(P|p)\.(M|m)\.", " PM ", text) # P\.M\.
    text = re.sub(r"(?<!\S)(N|n)\.(H|h)\.", "NH", text) # ng hapon
    text = re.sub(r"(?<!\S)(V|v)ol\.", "Vol", text) # Volume
    text = re.sub(r"(?<!\S)(A|a)tbp\.", "atbp", text) # At iba pa
    text = re.sub(r"(?<!\S)(I|i)bp\.", "ibp", text) # Iba pa
    text = re.sub(r"(?<!\S)(C|c)r\.", "credits", text) # Credits

    text = re.sub(r"(?<!\S)(S|s)(n)*r\.", "Snr", text) # Senior
    text = re.sub(r"(?<!\S)(J|j)(n)*r\.", "Jnr", text) # Junior
    text = re.sub(r"(?<!\S)(S|s)r\.", "Sr", text) # Senior
    text = re.sub(r"(?<!\S)(J|j)r\.", "Jr", text) # Junior
    text = re.sub(r"(?<!\S)(H|h)on\.", "Hon", text) # Honorable
    text = re.sub(r"(?<!\S)(M|m)essrs\.", "Messrs\.", text) # Plural of Mr
    text = re.sub(r"(?<!\S)(M|m)mes\.","Mmes\.", text) # Plural of Mrs
    text = re.sub(r"(?<!\S)(M|m)x\.","", text)

    text = re.sub(r"(?<!\S)(A|a)tty\.", "Atty", text) # Atty
    text = re.sub(r"(?<!\S)(C|c)ong\.", "Cong", text) # Congress(wo)man
    text = re.sub(r"(?<!\S)(P|p)res\.", "Pres", text) # President 
    text = re.sub(r"(?<!\S)(D|d)ra\.", "Dra", text) # Doktora
    text = re.sub(r"(?<!\S)(D|d)r\.", "Dr", text) # Doctor 
    text = re.sub(r"(?<!\S)(S|s)ec\.", "Sec", text) # Secretary
    text = re.sub(r"(?<!\S)(G|g)en\.", "Gen", text) # General
    text = re.sub(r"(?<!\S)(H|h)en\.", "Hen", text) # Heneral
    text = re.sub(r"(?<!\S)(S|s)en\.", "Sen", text) # Senator
    text = re.sub(r"\s(M|m)a\\.", "Ma\.", text) # Maria
    text = re.sub(r"(?<!\S)(E|e)ngr\.", "Engr", text) # Engineer
    text = re.sub(r"(?<!\S)(A|a)rch\.", "Arch", text) # Architect
    text = re.sub(r"(?<!\S)(R|r)ev\.", "Rev", text) # Reverend
    text = re.sub(r"(?<!\S)(F|f)r\.", "Fr", text) # father
    text = re.sub(r"(?<!\S)(B|b)r\.", "Br", text) # brother
    text = re.sub(r"(?<!\S)(A|a)ssoc\.", "Assoc", text) # Associate
    
    text = re.sub(r"(?<!\S)(M|m)r\.", "Mr", text) # Mister
    text = re.sub(r"(?<!\S)(M|m)rs\.", "Mrs", text) # Misis
    text = re.sub(r"(?<!\S)(M|m)s\.", "Ms", text) # Miss
    text = re.sub(r"G\.", "G", text) # Ginoong
    text = re.sub(r"Gg\.", "Gg", text) # Ginoong
    text = re.sub(r"(?<!\S)(G|g)ng\.", "Gng", text) # Ginang
    text = re.sub(r"(?<!\S)(B|b)b\.", "Bb", text) # Binibining
    text = re.sub(r"(?<!\S)(M|m)x\.", "Mx", text) # Gender neutral honorific
    text = re.sub(r"(?<!\S)(P|p)rof\.", "Prof", text) # Professor
    text = re.sub(r"(?<!\S)(D|d)oc\.", "Doc", text) # emergency room
    text = re.sub(r"E\.R\.", "ER", text) # emergency room
    text = re.sub(r"(?<!\S)(B|b)rgy\.", "Brgy", text) #barangay
    text = re.sub(r"\s(S|s)t\.", "St", text) # street and saint
    text = re.sub(r"(?<!\S)(S|s)ra\.", "Sra", text) # senora
    text = re.sub(r"(?<!\S)(C|c)or\.", "Cor", text) # Cor
    text = re.sub(r"(?<!\S)(S|s)ta\.", "Sta", text) # Santa
    
    text = re.sub(r"a\.d\.", "ad", text) 
    text = re.sub(r"b\.c\.e\.", "bce", text) 
    text = re.sub(r"b\.c\.", "bc", text) 
    text = re.sub(r"et al\.", "et al", text) # et al
    text = re.sub(r"e\.g\.", "eg", text) # e\.g\.
    text = re.sub(r"i\.e\.", "ie", text)  # i\.e\.
    text = re.sub(r"(?<!\S)(E|e)x\.", "ex", text)  # ex\.
    text = re.sub(r"approx\.", "approx", text) # approximately
    text = re.sub(r"(?<!\S)(A|a)ve\.", "Ave", text) # avenue
    text = re.sub(r"(?<!\S)(B|b)lvd\.", "Blvd", text) # boulevard
    text = re.sub(r"(?<!\S)(C|c)apt\.", "Capt", text) # captain
    text = re.sub(r"(?<!\S)(C|c)ol\.", "Col", text) # colonel
    text = re.sub(r"(?<!\S)(L|l)t\.", "Lt", text) # lieutenant
    text = re.sub(r"(?<!\S)(P|p)hd\.", "Phd", text) # Phd
    text = re.sub(r"(?<!\S)(S|s)gt\.", "Sgt", text) # sergeant
    text = re.sub(r"(?<!\S)(R|r)rd\.", "Rd", text) # roadd
    text = re.sub(r"(?<!\S)(S|s)gt\.", "Sgt", text) # sergeant
    text = re.sub(r"(?<!\S)(S|s)ra\.", "Sra", text) # senora

    text = re.sub(r"En\.", "En", text) # Enero
    text = re.sub(r"Peb\.", "Peb", text) # Pebrero
    text = re.sub(r"Mar\.", "Mar", text) # Marso
    text = re.sub(r"Abr\.", "Abr", text) # Abril
    text = re.sub(r"May\.", "May", text) # Mayo
    text = re.sub(r"Hun\.", "Hun", text) # Hunyo
    text = re.sub(r"Hul\.", "Hul", text) # Hulyo
    text = re.sub(r"Ag\.", "Ag", text) # Agosto
    text = re.sub(r"Set\.", "Set", text) # Setyembre
    text = re.sub(r"Okt\.", "Okt", text) # Oktubre
    text = re.sub(r"Nob\.", "Nob", text) # Nobyembre
    text = re.sub(r"Dis\.", "Dis", text) # Disyembre

    text = re.sub(r"Mon\.", "Mon", text) #Monday
    text = re.sub(r"Tues\.", "Tues", text) #Tuesday
    text = re.sub(r"Wed\.", "Wed", text) #Wednesday
    text = re.sub(r"Thur\.", "Thur", text) #Thursday
    text = re.sub(r"Fri\.", "Fri", text) #Friday
    text = re.sub(r"Thur\.", "Thur", text) #Thursday
    text = re.sub(r"Fri\.", "Fri", text) #Friday
    text = re.sub(r"Sat\.", "Sat", text) #Saturday
    text = re.sub(r"Sun\.", "Sun", text) #Sunday

    return text.strip()

def stripUnicode(text):
    text_encoded= text.encode("ascii", "ignore")
    text_decoded= text_encoded.decode()
    
    return text_decoded

def clean_before_sentence_split(text):
    result = html.unescape(html.unescape(str(text)))
    result = normalize_abbreviations(result)
    #result = demoji.replace(result, " XX_EMOJI ")
    result = stripUnicode(result) # alternative to demoji
    result = re.sub(r"([\r\n\t\f\v]+( )*)+", " ", result) # XX_DELIMITER
    result = re.sub(r"https?:\/\/([\w\-_]+\.)+([\w\-_]+)+(\/[^\s]+)*", " XX_URL ", result) #URL
    result = re.sub(r"[\SÑñ]+@([\SÑñ]+\.)+[\SÑñ]+", " XX_EMAIL ", result) # EMAIL
    result = re.sub(r"([\w\-_]+\.)+(com|net|org|co|us|ph|io)(\/[^\s]+)*", " XX_URL ", result, flags=re.IGNORECASE) # URL
    result = re.sub(r"(?<!\S)@[^\s.,!?]+(?!\S)", " XX_USERNAME ", result)
    result = re.sub(r"(?<!\S)#[a-zA-Z_][a-zA-ZÑñ0-9_]*(?!\S)", " XX_HASHTAG ", result) # HASHTAG
    result = re.sub(r"(?<!\S)([^a-zA-Z0-9\s])\1+(?!\S)", " XX_SEQSAMESYMBOLS ", result) # XX_SEQSAMESYMBOLS
    result = re.sub(r"(?<!\S)[^a-zA-Z0-9\s][^a-zA-Z0-9\s]+(?!\S)", " XX_SEQNOTSAMESYMBOLS ", result) # XX_SEQNOTSAMESYMBOLS
    result = re.sub(r"(?<![0-9]):[03\(\)DPO]+", " XX_SEQNOTSAMESYMBOLS ", result) # :), :(, :3, :0, :D, :P, :O --> XX_SEQNOTSAMESYMBOLS

    # normalize quotes
    result = re.sub(r'“', '"', result)
    result = re.sub(r'”', '"', result)
    result = re.sub(r"‘", "'", result)
    result = re.sub(r"’", "'", result)

    result = re.sub(r",+", " , ", result)
    result = re.sub(r"\?+", " ? ", result)
    result = re.sub(r"!+", " ! ", result)
    result = re.sub(r"\.\.+", " ... ", result) # ellipsis 
    result = re.sub(r"…+", " ... ", result) # ellipsis
    result = re.sub(r"(?<![0-9]):+", " : ", result) # colon not preceded by numbers (to preserve time format)
    result = re.sub(r";+", " ; ", result)
    result = re.sub(r"%+", " % ", result)
    result = re.sub(r"#+", " # ", result)
    result = re.sub(r"[\[\(\<\{]", " ( ", result)
    result = re.sub(r"[\]\)\>\}]", " ) ", result)
    result = re.sub(r"/+", " / ", result)

    result = re.sub("((?<=\w)[-−‒–—]{2,}(?=\w))", r" \1 ", result)
    result = re.sub("([-−‒–—]+(?!\w)|(?<!\w)[-−‒–—]+)", r" \1 ", result)
    result = re.sub("((?<=\w)[−–—](?=\w))", r" \1 ", result)

    result = re.sub(r"@[A-Za-z0-9_]+", " XX_USERNAME ", result) # catch all for usernames

    # final cleaning
    result = re.sub(r'\s\s+',' ', result) # replace 2 or more whitespaces

    
    # result = re.sub(r'^[\d]+|(?<=.)\d+','', result) # for bible, remove numbers before a sentence

    return result.strip()

def clean_after_sentence_split(text):
    #result = re.sub(r'(?<!XX)_',' ', text)
    result = re.sub(r"(?<![^A-Za-z0-9])\.", " . ", text) # dot not preceded by another dot non-alphanumeric
    result = re.sub(r'\s\s+',' ', result) # replace 2 or more whitespaces with 1 whitespace
    result_no_special_toks = re.sub(r'(?<!\S)XX_[A-Za-z]*(?!\S)','', result) # remove all XX_*
    result_no_special_toks = re.sub(r'\s\s+',' ', result_no_special_toks) # replace 2 or more whitespaces with 1 whitespace FOR result_POSTag


    return result.strip(), result_no_special_toks.strip()

def process_sentences(text_list):
    # variables
    sentence_list = []

    # clean before sentence segmentation
    start = time.time()
    docs = [clean_before_sentence_split(text) for text in text_list]
    print(f"clean_before_sentence_split ---{time.time() - start} seconds---")

    # sentence segmentation
    start = time.time()
    # for doc in nlp.pipe(docs, n_process=2, batch_size=10000):
    #     for sentence in doc.sents:
    #         sentence_list.append(sentence.text)
    sentence_list = [sent for doc in docs for sent in tokenizer.tokenize(doc)]
    print(f"sentence segmentation ---{time.time() - start} seconds---")
    

    # clean after sentence segmentation (returns tuple)
    start = time.time()
    sentence_list = np.array([clean_after_sentence_split(text) for text in sentence_list])
    print(f"sentence clean_after_sentence_split ---{time.time() - start} seconds---")

    return sentence_list[:,0], sentence_list[:,1]

# may fil and eng modes
def preprocess(text_list, partition_id='0', save_path="data.json"):
    # tokenize by sentences and preprocess each sentence
    print("cleaning texts...")
    start = time.time()
    sentences_with_special_toks, sentences_no_special_toks = process_sentences(text_list)
    print(f"---{time.time() - start} seconds---")

    print(f"n_sentences: {sentences_no_special_toks.shape}")

    # language tagging
    print("running language detection...")
    start = time.time()
    detector = LanguageIdentification()
    langs, probs = detector.predict_lang_list(sentences_no_special_toks)
    detector_results = pd.DataFrame({'lang':pd.Series(langs, dtype='str'),
                   'probs':pd.Series(probs, dtype='float32')})

    # filter sentences by language
    threshold = -999
    fil_idxs = detector_results[(detector_results['lang'] == 'tl') & (detector_results['probs'] >= threshold)].index.values
    en_idxs = detector_results[(detector_results['lang'] == 'en') & (detector_results['probs'] >= threshold)].index.values

    print(f"FIL: {len(fil_idxs)}, ENG: {len(en_idxs)}")
    print(f"---{time.time() - start} seconds---")

    # pos tag FIL
    print("POS tagging FIL")
    start = time.time()
    fil_tags, fil_pos = batch_tag_fil(sentences_no_special_toks[fil_idxs], bs=2048)
    print(f"---{time.time() - start} seconds---")

    # pos tag EN
    print("POS tagging ENG")
    start = time.time()
    en_tags, en_pos = batch_tag_eng(sentences_no_special_toks[en_idxs])
    print(f"---{time.time() - start} seconds---")

    # build schema
    print("building schema...")
    start = time.time()
    col_probs = np.concatenate([detector_results.loc[fil_idxs]['probs'].to_numpy(), detector_results.loc[en_idxs]['probs'].to_numpy()])
    print("1")
    
    df = pd.DataFrame({'text': np.concatenate([sentences_with_special_toks[fil_idxs], sentences_with_special_toks[en_idxs]]), 'pos_tags': np.concatenate([fil_tags, en_tags]), 'lang': np.concatenate([['fil']*len(fil_idxs), ['en']*len(en_idxs)]), 'lang_prob': col_probs, 'source': source, 'source_type': source_type, 'year': year, 'month': month})

    df_pos = pd.DataFrame({'text': np.concatenate([sentences_with_special_toks[fil_idxs], sentences_with_special_toks[en_idxs]]), 'pos_tags': fil_pos + en_pos})
    
    print("2")
    ## drop short sentences
    sentence_drop_thres = 4
    print(f"---dropping sentences with tokens <= {sentence_drop_thres} (punctuations excluded from count)..")
    mask = (df['text'].str.replace("[^A-Za-z0-9\s]", "").str.split(' ').str.len() > sentence_drop_thres) # exclude strings with 3 tokens or less
    df = df.loc[mask]
    df.reset_index(drop=True, inplace=True)

    # mask_pos = (df_pos['text'].str.replace("[^A-Za-z0-9\s]", "").str.split(' ').str.len() > sentence_drop_thres) # exclude strings with 3 tokens or less
    df_pos = df_pos.loc[mask]
    df_pos.reset_index(drop=True, inplace=True)
    df_pos = df_pos['pos_tags'].to_frame()

    print(f'df: {df.size}')
    print(f'df_pos: {df_pos.size}')
    # print("---counting tokens...")
    # df['token_count'] = df['text'].str.split(' ').str.len()

    # save
    print(f'Saving {save_path}...')
    display(df)
    display(df_pos)
    # concat NETSCI version of POS
    df['pos_tag_2'] = df_pos
    df.to_json(path_or_buf=f"{save_path}.json", orient="records")
    df_pos.to_json(path_or_buf=f"{save_path}_POS.json", orient="records")
    print(f"---{time.time() - start} seconds---")

    return df

## Single file preprocessing

In [ ]:
## OPTION TO LOAD SPECIFIC MONTH ONLY (ideal for big data like Twitter data)

# Go to Shared GDrive directory containing data
base = "/content/drive/Shareddrives/DOST FilWordNet x WordSense/Thesis/Data Collection/RAW_COHFIE"
source_type = 'online_forums'  #change
source = 'reddit'    #change
year = 2021
month = 10
path = f'{base}/{source_type}/{source}/{year}/{month}'

# Load all .json

# the path to your csv file directory
data_dir = path

# get all the csv files in that directory (assuming they have the extension .csv)
json_files = glob.glob(os.path.join(data_dir, f'{month}.json'))

# loop through the files and read them in with pandas
dataframes = []  # a list to hold all the individual pandas DataFrames
for file in json_files:
    df = pd.read_json(file, orient='records')
    dataframes.append(df)

# concatenate them all together
df = pd.concat(dataframes, ignore_index=True)

display(df)

# set field
text_column_name = 'text'

# Get number of rows
n_rows = df.shape[0]

# Set n_partitions
if n_rows > 1000000:
    n_partitions = 20
elif n_rows > 500000:
    n_partitions = 10
elif n_rows > 300000:
    n_partitions = 6
elif n_rows > 10:
    n_partitions = 4
else:
    n_partitions = 1

# Generate partitions
print(f"n_partitions: {n_partitions}")
partitions = np.array_split(range(n_rows), n_partitions)

print(f"PROCESSING {path}")

start = time.time()
for partition_id, partition in enumerate(partitions):
    if partition_id > 4:
      print(f"===PARTITION {partition_id+1}/{n_partitions}===")

      PREPROC_COHFIE_BASE = f"/content/drive/Shareddrives/DOST FilWordNet x WordSense/Thesis/Data Collection/COHFIE/{source_type}/{source}/{year}/{month}"
      FILENAME = f"{source_type}_{source}_{year}_{month}_{partition_id}"
      SAVE_PATH = f'{PREPROC_COHFIE_BASE}/{FILENAME}'
      try:
          if not os.path.exists(PREPROC_COHFIE_BASE):
              os.makedirs(PREPROC_COHFIE_BASE)
          %time res = preprocess(text_list=df[text_column_name].loc[partition].tolist(), partition_id=partition_id, save_path=SAVE_PATH)
          display(res)
      except Exception as e:
          if len(os.listdir(PREPROC_COHFIE_BASE)) == 0 and partition_id == 0: #delete folder if empty
            os.rmdir(PREPROC_COHFIE_BASE)
          print(e)
          pass


print(f"TOTAL TIME: ---{time.time() - start} seconds---")

,author,created_utc,full_link,text,subreddit,title,created
0,AutoModerator,2021-10-01 00:00:14,https://www.reddit.com/r/beautytalkph/comments...,Need help with skincare? What's the difference...,beautytalkph,"Skincare Thread | October 01, 2021",1633017614
1,AutoModerator,2021-10-01 00:00:15,https://www.reddit.com/r/FilmClubPH/comments/p...,"So, what have you watched recently?",FilmClubPH,Monthly /r/FilmClubPH Discussion Thread - Octo...,1633017615
2,sajsd,2021-10-01 00:00:31,https://www.reddit.com/r/phr4r/comments/pymsgw...,Been feeling quite burnt-out lately due to aca...,phr4r,18 [M4M] Anyone up for a quick breather ;),1633017631
3,AutoModerator,2021-10-01 00:01:13,https://www.reddit.com/r/beautytalkph/comments...,Are you a frag head? Want a recommendation for...,beautytalkph,"Fragrance Thread | October 01, 2021",1633017673
4,gurlismmmme,2021-10-01 00:01:41,https://www.reddit.com/r/phclassifieds/comment...,"Hello, my budget is less than 5k. Second hand ...",phclassifieds,Looking for second-hand IPS monitor 22-24”,1633017701
...,...,...,...,...,...,...,...
18558,baejaeta,2021-10-31 23:55:21,https://www.reddit.com/r/phinvest/comments/qjs...,"Just a fun sharing, if you can share your curr...",phinvest,Can you share your credit card - credit limit ...,1635695721
18559,NgokNgak,2021-10-31 23:55:29,https://www.reddit.com/r/phr4r/comments/qjse7t...,Ayun meron ka bang naughty ans spooky experien...,phr4r,27 [M4F] HAPPY HALLOWEEN! SHARE YOUR KWENTO,1635695729
18560,LoneWolf899,2021-10-31 23:56:35,https://www.reddit.com/r/phr4r/comments/qjsf15...,Looking for a girl : \- I can go on dates with...,phr4r,26 [M4F] FWB,1635695795
18561,mariyahuwana,2021-10-31 23:57:23,https://www.reddit.com/r/phr4r/comments/qjsfkh...,Thanks for the past invites for Halloween part...,phr4r,28 [F4A] MISSING HOELLOWEEN CROWD!,1635695843


n_partitions: 4
PROCESSING /content/drive/Shareddrives/DOST FilWordNet x WordSense/Thesis/Data Collection/RAW_COHFIE/online_forums/reddit/2021/10
===PARTITION 1/4===
cleaning texts...
clean_before_sentence_split ---7.861918210983276 seconds---
sentence segmentation ---0.6655018329620361 seconds---
sentence clean_after_sentence_split ---0.9638745784759521 seconds---
---9.492334365844727 seconds---
n_sentences: (24082,)
running language detection...


FIL: 8846, ENG: 13347
---1.0577874183654785 seconds---
POS tagging FIL
Progress: 23.151706986208456%
Progress: 46.30341397241691%
Progress: 69.45512095862537%
Progress: 92.60682794483382%
Progress: 100.0%
---5.341285943984985 seconds---
POS tagging ENG
---15.809266328811646 seconds---
building schema...
1
2
---dropping sentences with tokens <= 4 (punctuations excluded from count)..
df: 156280
df_pos: 19535
Saving /content/drive/Shareddrives/DOST FilWordNet x WordSense/Thesis/Data Collection/COHFIE/online_forums/reddit/2021/10/online_forums_reddit_2021_10_0...


/usr/local/lib/python3.7/dist-packages/ipykernel_launcher.py:241: FutureWarning: The default value of regex will change from True to False in a future version.


,text,pos_tags,lang,lang_prob,source,source_type,year,month
0,Need help with skincare ?,FW FW FW FW PMQ,fil,-1.000000,reddit,online_forums,2021,10
1,Are you a frag head ?,NNP NNP NNP NNP NNP PMQ,fil,-1.000000,reddit,online_forums,2021,10
2,Want a recommendation for a new signature scent ?,FW NNP NNP FW NNP NNP NNP NNP PMQ,fil,-1.000000,reddit,online_forums,2021,10
3,Ask any questions about fragrances here !,NNP NNP NNP NNP NNP NNP PMS,fil,-1.000000,reddit,online_forums,2021,10
4,May kailangan kang ilabas na sama ng loob ?,VBH VBS FW VBOF CCP NNC CCB NNC PMQ,fil,0.702209,reddit,online_forums,2021,10
...,...,...,...,...,...,...,...,...
19530,I seriously want to know what is and would be ...,PRP RB VBP TO VB WP VBZ CC MD VB NNP NNP POS N...,en,0.993061,reddit,online_forums,2021,10
19531,I personally hope she would be more critical l...,PRP RB VBP PRP MD VB RBR JJ IN WP NNP VBZ VBN VBG,en,0.993432,reddit,online_forums,2021,10
19532,i'm a student in UPD and i have this class ( a...,NN VBP DT NN IN NNP CC RB VBP DT NN VBP RB VBG...,en,0.865745,reddit,online_forums,2021,10
19533,"Buti pa ibang classes ko , very concerned sa w...",NNP NN NN NNS VBP RB JJ NN VBG JJ NN VBD NNS I...,en,0.899599,reddit,online_forums,2021,10


...

,text,pos_tags,lang,lang_prob,source,source_type,year,month
0,"im still a V , so NO SEX here ... bye wag na i...",NNP NNP NNP CDB PMC NNP FW FW NNP PMS NNP NNP ...,fil,-1.000000,reddit,online_forums,2021,10
1,Ang korni pero di ko alam bakit nagpplay sa ut...,DTC NNC CCT RBF PRS VBS RBD_CCP VBTR CCT NNC P...,fil,0.706677,reddit,online_forums,2021,10
2,"Anyway , baka may forever dito or baka nandito...",FW PMC NNC_CCP VBH JJD PRL CCT NNC_CCP VBTS DT...,fil,0.856481,reddit,online_forums,2021,10
3,About me : - Around 5'1 - from Big 4 - Profess...,NNP NNP PMS PMS NNP NNP PMS NNP NNP CDB PMS CD...,fil,-1.000000,reddit,online_forums,2021,10
4,Minsan sa tagal na wala akong nakakusap o naya...,RBW CCT NNC CCP VBN PRS_CCP FW CCT NNC PMC RBF...,fil,0.822286,reddit,online_forums,2021,10
...,...,...,...,...,...,...,...,...
21230,Any rank / elo is welcome but I would apprecia...,DT NN NN NN VBZ JJ CC PRP MD VB PRP IN PRP VBP...,en,0.967800,reddit,online_forums,2021,10
21231,If you're a group of friends who are just star...,IN PRP VBP DT NN IN NNS WP VBP RB VBG RP CC VB...,en,0.974067,reddit,online_forums,2021,10
21232,Let's just see to it that there's a slot for m...,VB POS RB VB TO PRP IN EX VBZ DT NN IN PRP CC ...,en,0.987128,reddit,online_forums,2021,10
21233,The only requirement I would ask is for you / ...,DT JJ NN PRP MD VB VBZ IN PRP VBP PRP VBP TO R...,en,0.983174,reddit,online_forums,2021,10


,pos_tags
0,"[(im, NNP), (still, NNP), (a, NNP), (V, CDB), ..."
1,"[(Ang, DTC), (korni, NNC), (pero, CCT), (di, R..."
2,"[(Anyway, FW), (,, PMC), (baka, NNC_CCP), (may..."
3,"[(About, NNP), (me, NNP), (:, PMS), (-, PMS), ..."
4,"[(Minsan, RBW), (sa, CCT), (tagal, NNC), (na, ..."
...,...
21230,"[(Any, DT), (rank, NN), (/, NN), (elo, NN), (i..."
21231,"[(If, IN), (you, PRP), ('re, VBP), (a, DT), (g..."
21232,"[(Let, VB), ('s, POS), (just, RB), (see, VB), ..."
21233,"[(The, DT), (only, JJ), (requirement, NN), (I,..."


---1.1578357219696045 seconds---
CPU times: user 29.1 s, sys: 981 ms, total: 30 s
Wall time: 30 s


,text,pos_tags,lang,lang_prob,source,source_type,year,month,pos_tag_2
0,"im still a V , so NO SEX here ... bye wag na i...",NNP NNP NNP CDB PMC NNP FW FW NNP PMS NNP NNP ...,fil,-1.000000,reddit,online_forums,2021,10,"[(im, NNP), (still, NNP), (a, NNP), (V, CDB), ..."
1,Ang korni pero di ko alam bakit nagpplay sa ut...,DTC NNC CCT RBF PRS VBS RBD_CCP VBTR CCT NNC P...,fil,0.706677,reddit,online_forums,2021,10,"[(Ang, DTC), (korni, NNC), (pero, CCT), (di, R..."
2,"Anyway , baka may forever dito or baka nandito...",FW PMC NNC_CCP VBH JJD PRL CCT NNC_CCP VBTS DT...,fil,0.856481,reddit,online_forums,2021,10,"[(Anyway, FW), (,, PMC), (baka, NNC_CCP), (may..."
3,About me : - Around 5'1 - from Big 4 - Profess...,NNP NNP PMS PMS NNP NNP PMS NNP NNP CDB PMS CD...,fil,-1.000000,reddit,online_forums,2021,10,"[(About, NNP), (me, NNP), (:, PMS), (-, PMS), ..."
4,Minsan sa tagal na wala akong nakakusap o naya...,RBW CCT NNC CCP VBN PRS_CCP FW CCT NNC PMC RBF...,fil,0.822286,reddit,online_forums,2021,10,"[(Minsan, RBW), (sa, CCT), (tagal, NNC), (na, ..."
...,...,...,...,...,...,...,...,...,...
21230,Any rank / elo is welcome but I would apprecia...,DT NN NN NN VBZ JJ CC PRP MD VB PRP IN PRP VBP...,en,0.967800,reddit,online_forums,2021,10,"[(Any, DT), (rank, NN), (/, NN), (elo, NN), (i..."
21231,If you're a group of friends who are just star...,IN PRP VBP DT NN IN NNS WP VBP RB VBG RP CC VB...,en,0.974067,reddit,online_forums,2021,10,"[(If, IN), (you, PRP), ('re, VBP), (a, DT), (g..."
21232,Let's just see to it that there's a slot for m...,VB POS RB VB TO PRP IN EX VBZ DT NN IN PRP CC ...,en,0.987128,reddit,online_forums,2021,10,"[(Let, VB), ('s, POS), (just, RB), (see, VB), ..."
21233,The only requirement I would ask is for you / ...,DT JJ NN PRP MD VB VBZ IN PRP VBP PRP VBP TO R...,en,0.983174,reddit,online_forums,2021,10,"[(The, DT), (only, JJ), (requirement, NN), (I,..."


TOTAL TIME: ---125.95753931999207 seconds---


## Iterate directory

In [ ]:
# Go to Shared GDrive directory containing data
base = "/content/drive/Shareddrives/DOST FilWordNet x WordSense/Thesis/Data Collection/RAW_COHFIE"
source_type = 'books'  #change
source = '100year'    #change
path = f'{base}/{source_type}/{source}/'
years = next(os.walk(f'{path}'))[1]

# set field
text_column_name = 'text'

# directory iterate from year down to months 
for year in years:
    print(f"PROCESSING: YEAR {year}")
    months = next(os.walk(f'{path}/{year}'))[1]

    for month in months:
        print(f"PROCESSING: MONTH {month}")

        data_path = f'{path}/{year}/{month}'
        print(f"PROCESSING {data_path}")

        # get all the csv files in that directory (assuming they have the extension .json)
        json_files = glob.glob(f'{data_path}/{month}.json')

        # loop through the files and read them in with pandas
        dataframes = []  # a list to hold all the individual pandas DataFrames
        for file in json_files:
            df = pd.read_json(file, orient='records')
            print("PREVIEW:")
            display(df.iloc[:3])
            # df = pd.read_json(file, orient='records')
            # remove empty columns
            dataframes.append(df)

        # concatenate them all together
        df = pd.concat(dataframes, ignore_index=True)

        # Get number of rows
        n_rows = df.shape[0]
        print(f"n_rows: {n_rows}")

        # Set n_partitions
        if n_rows > 1000000:
            n_partitions = 20
        elif n_rows > 500000:
            n_partitions = 10
        elif n_rows > 300000:
            n_partitions = 6
        elif n_rows > 10:
            n_partitions = 4
        else:
            n_partitions = 1

        # Generate partitions
        print(f"n_partitions: {n_partitions}")
        partitions = np.array_split(range(n_rows), n_partitions)

        # call preprocess function per partition
        start = time.time()
        for partition_id, partition in enumerate(partitions):
            print(f"===PARTITION {partition_id}===")
            PREPROC_COHFIE_BASE = f"/content/drive/Shareddrives/DOST FilWordNet x WordSense/Thesis/Data Collection/COHFIE/{source_type}/{source}/{year}/{month}"
            FILENAME = f"{source_type}_{source}_{year}_{month}_{partition_id}"
            SAVE_PATH = f'{PREPROC_COHFIE_BASE}/{FILENAME}'
            try:
                if not os.path.exists(PREPROC_COHFIE_BASE):
                    os.makedirs(PREPROC_COHFIE_BASE)
                %time res = preprocess(text_list=df[text_column_name].loc[partition].tolist(), partition_id=partition_id, save_path=SAVE_PATH)
                display(res)
            except Exception as e:
                if len(os.listdir(PREPROC_COHFIE_BASE)) == 0 and partition_id == 0: #delete folder if empty
                  os.rmdir(PREPROC_COHFIE_BASE)
                print(e)
                pass

        print(f"TOTAL TIME: ---{time.time() - start} seconds---")

Output hidden; open in https://colab.research.google.com to view.

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=184a8086-5bdb-495f-b7b4-18183b5b22b7' target="_blank">
 </img>
Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>